# Trayectorias temporales — MapBiomas Colombia

Detección y corrección de trayectorias LULC anómalas.
Ver README.md. Detalle operativo local: EXPLICACION.txt.


## Método

Detalle en EXPLICACION.txt.


## 1. Conexión a Earth Engine


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import ee

GEE_PROJECT = 'mapbiomas-colombia'

if 'ee_initialized' not in globals():
    ee.Initialize(project=GEE_PROJECT)
    ee_initialized = True
    print(f'Conectado a GEE — proyecto: {GEE_PROJECT}')
else:
    print('GEE ya inicializado en esta sesión.')

## 2. Configuración


In [ ]:
import re
import json
from pathlib import Path
import pandas as pd

FOLDER_CLASS = 'projects/mapbiomas-colombia/assets/LULC/COLECCION4/clasificacion'
FOLDER_FT = 'projects/mapbiomas-colombia/assets/LULC/COLECCION4/clasificacion-ft'
FOLDER = FOLDER_FT  # FOLDER_FT | FOLDER_CLASS
REGION_VECTOR = 'projects/mapbiomas-colombia/assets/DATOS_AUXILIARES/VECTORES/clasificacion-regiones-3-buffer-250m'
from paths import OUTPUTS, LEYENDA, REGIONES, CSV_ANOMALIAS
REGIONES_XLSX = REGIONES
REGION_ID = 30450  # None = nacional (sin mapa/stats); int/str = una región (ej. 30450)
VERSION_INPUT = 7  # solo con REGION_ID: 1 = clasificacion; >1 = filtros; None = máx en FT (excl. 11, 99)
REEXPORTAR = False  # False | True (regenera CSV de anomalías)
OUTPUT_CSV = CSV_ANOMALIAS
OUTPUTS.mkdir(exist_ok=True)
TOTAL_PIXELES_COLOMBIA_30M = 1_283_824_234
YEARS = ee.List.sequence(1986, 2023)
CLASE_BOSQUE = 3
CLASES_PLANTACION = (9, 35, 74)  # silvicultura, palma, banano

# Mayor → menor prevalencia. Ejecutar reversed() en huecos.
# Bloques: A infra | B costa/agua | C inundables | D sustratos | E agro | F residual | G natural
PRIORIDAD_CLASES = (
    34, 30, 75, 24,          # A
    32, 23, 5, 31, 33,       # B
    11, 6, 82,               # C
    29, 68,                  # D
    9, 40, 74, 35,           # E
    21, 25,                  # F
    81, 13, 12, 3, 49, 50,   # G
)
VENTANAS_HUECOS = (3, 4, 5)  # longitudes de hueco A-X-A / A-XX-A / A-XXX-A (modo 'todas')
CORREGIR_BORDES = False  # False | True (extiende clases a extremos de la serie)
PASADAS_BOSQUE_RESIDUAL = 2  # nº de pasadas bosque↔plantación
MODO_CORRECCION = 'Todas'  # 'bosque' | 'todas'

MAPA_ANOMALIAS = 'todas'  # 'bosque' | 'bosque_plantacion' | 'todas'

LEYENDA_PATH = LEYENDA
with open(LEYENDA_PATH, encoding='utf-8') as f:
    _leyenda = json.load(f)
CLASS_NAMES = {int(k): v for k, v in _leyenda['class_names'].items()}
PALETTE_LULC = _leyenda['palette_lulc']

_FUENTE_PRIORIDAD = {
    34: 'A transversal CO',
    30: 'A transversal CO/BR',
    75: 'A infra CO',
    24: 'A infra CO/BR',
    32: 'B costa CO/BR',
    23: 'B costa CO/BR',
    5: 'B costa BR + exc. CO',
    31: 'B acuático BR',
    33: 'B agua CO',
    11: 'C inundable exc. CO',
    6: 'C inundable exc. CO',
    82: 'C inundable andina',
    29: 'D sustrato CO/BR',
    68: 'D sustrato CO',
    9: 'E agro específico CO/BR',
    40: 'E agro temporal BR',
    74: 'E agro perenne CO',
    35: 'E agro perenne CO/BR',
    21: 'F residual CO/BR',
    25: 'F residual CO',
    81: 'G natural CO',
    13: 'G natural CO',
    12: 'G natural CO',
    3: 'G bosque CO/BR',
    49: 'G arena CO',
    50: 'G arena CO',
}
MATRIZ_IMPORTANCIA = pd.DataFrame([
    {
        'rango': i,
        'clase': cid,
        'nombre': CLASS_NAMES.get(cid, f'Clase {cid}'),
        'bloque': _FUENTE_PRIORIDAD.get(cid, 'revisar'),
    }
    for i, cid in enumerate(PRIORIDAD_CLASES, start=1)
])
print(f'Matriz de importancia: {len(PRIORIDAD_CLASES)} clases de nivel 2')
print(f"Mapa anomalías: {MAPA_ANOMALIAS}")
print(f'REGION_ID={REGION_ID} · VERSION_INPUT={VERSION_INPUT}')
display(MATRIZ_IMPORTANCIA)

VIS_ANOMALIAS = {'min': 1, 'max': 4, 'palette': ['#fee08b', '#fdae61', '#f46d43', '#a50026']}
LEGEND_ANOMALIAS = {
    'Bosque 1': 'fee08b',
    'Bosque 2': 'fdae61',
    'Bosque 3': 'f46d43',
    'Bosque 4+': 'a50026',
}
VIS_ANOMALIAS_PLANT = {'min': 1, 'max': 4, 'palette': ['#a5d8ff', '#4dabf7', '#228be6', '#1864ab']}
LEGEND_ANOMALIAS_PLANT = {
    'Plantacion 1': 'a5d8ff',
    'Plantacion 2': '4dabf7',
    'Plantacion 3': '228be6',
    'Plantacion 4+': '1864ab',
}
LEGEND_ANOMALIAS_OTRAS = {
    'Otras 1': 'd6bcfa',
    'Otras 2': 'b794f4',
    'Otras 3': '9f7aea',
    'Otras 4+': '553c9a',
}
VIS_ANOMALIAS_OTRAS = {'min': 1, 'max': 4, 'palette': ['#d6bcfa', '#b794f4', '#9f7aea', '#553c9a']}
LEGEND_ANOMALIAS_MIXTO = {**LEGEND_ANOMALIAS, **LEGEND_ANOMALIAS_PLANT}
LEGEND_ANOMALIAS_TODAS = {**LEGEND_ANOMALIAS_MIXTO, **LEGEND_ANOMALIAS_OTRAS}
VIS_LULC = {'min': 0, 'max': len(PALETTE_LULC) - 1, 'palette': PALETTE_LULC}


## 3. Funciones


In [ ]:

def class_name(class_id):
    return CLASS_NAMES.get(int(class_id), f'Clase {int(class_id)}')


def lulc_color(class_id):
    idx = int(class_id)
    return PALETTE_LULC[idx] if 0 <= idx < len(PALETTE_LULC) else '#888888'


def band_name(year):
    return ee.String('classification_').cat(ee.Number(year).format('%d'))


def es_plantacion(img):
    """Silvicultura (9), palma (35) o plátano/banano (74)."""
    mask = img.eq(CLASES_PLANTACION[0])
    for cid in CLASES_PLANTACION[1:]:
        mask = mask.Or(img.eq(cid))
    return mask


def reemplazo_falso_bosque(prev, next1):
    rep = ee.Image(prev)
    for cid in CLASES_PLANTACION:
        rep = rep.where(next1.eq(cid), cid)
    return rep


def reemplazo_falso_bosque_2anio_ini(prev, next1, next2):
    rep = ee.Image(prev)
    for cid in CLASES_PLANTACION:
        rep = rep.where(next1.eq(cid), cid)
    seguido_de_bosque = next1.eq(CLASE_BOSQUE)
    for cid in CLASES_PLANTACION:
        rep = rep.where(seguido_de_bosque.And(next2.eq(cid)), cid)
    return rep


def anomalia_bosque(prev1, curr, next1, next2):
    error1 = prev1.neq(CLASE_BOSQUE).And(curr.eq(CLASE_BOSQUE)).And(next1.neq(CLASE_BOSQUE))
    error2_ini = prev1.neq(CLASE_BOSQUE).And(curr.eq(CLASE_BOSQUE)).And(next1.eq(CLASE_BOSQUE)).And(next2.neq(CLASE_BOSQUE))
    return error1.Or(error2_ini)


def anomalia_plantacion(prev1, curr, next1, next2):
    entre_1 = prev1.eq(CLASE_BOSQUE).And(es_plantacion(curr)).And(next1.eq(CLASE_BOSQUE))
    entre_2 = (
        prev1.eq(CLASE_BOSQUE)
        .And(es_plantacion(curr))
        .And(es_plantacion(next1))
        .And(next2.eq(CLASE_BOSQUE))
    )
    antes_1 = es_plantacion(prev1).Not().And(es_plantacion(curr)).And(next1.eq(CLASE_BOSQUE))
    antes_2 = (
        es_plantacion(prev1).Not()
        .And(es_plantacion(curr))
        .And(es_plantacion(next1))
        .And(next2.eq(CLASE_BOSQUE))
    )
    return entre_1.Or(entre_2).Or(antes_1).Or(antes_2)


def es_otra_clase(img):
    return img.neq(CLASE_BOSQUE).And(es_plantacion(img).Not())


def anomalia_otras(prev1, curr, next1, next2):
    # Clase ≠ bosque/plantación aislada 1–2 años (misma lógica que bosque).
    es_otra = es_otra_clase(curr)
    error1 = prev1.neq(curr).And(es_otra).And(next1.neq(curr))
    error2 = prev1.neq(curr).And(es_otra).And(next1.eq(curr)).And(next2.neq(curr))
    return error1.Or(error2)


def anomalia_mapa(prev1, curr, next1, next2, modo='bosque'):
    modo = str(modo or 'bosque').strip().lower()
    a = ee.Image(anomalia_bosque(prev1, curr, next1, next2)).unmask(0)
    if modo in ('bosque_plantacion', 'bosque+plantacion', 'todas') or 'plant' in modo:
        a = a.Or(ee.Image(anomalia_plantacion(prev1, curr, next1, next2)).unmask(0))
    if modo == 'todas' or 'otra' in modo:
        a = a.Or(ee.Image(anomalia_otras(prev1, curr, next1, next2)).unmask(0))
    return a.gt(0)


def describe_trayectoria(p1, c, n1, n2):
    p1, c, n1, n2 = map(int, (p1, c, n1, n2))
    plant = set(CLASES_PLANTACION)
    if c == CLASE_BOSQUE:
        if n1 != CLASE_BOSQUE and n2 != CLASE_BOSQUE:
            return 'Bosque aislado 1 año'
        if n1 == CLASE_BOSQUE and n2 != CLASE_BOSQUE:
            return 'Bosque aislado 2 años'
        return 'Revisar patrón'
    if c in plant:
        if p1 == CLASE_BOSQUE and n1 == CLASE_BOSQUE:
            return 'Plantación aislada 1 año'
        if p1 == CLASE_BOSQUE and n1 in plant and n2 == CLASE_BOSQUE:
            return 'Plantación aislada 2 años'
        if p1 not in plant and n1 == CLASE_BOSQUE:
            return 'Plantación corta antes de bosque 1 año'
        if p1 not in plant and n1 in plant and n2 == CLASE_BOSQUE:
            return 'Plantación corta antes de bosque 2 años'
        return 'Revisar plantación'
    if c != CLASE_BOSQUE and c not in plant:
        if n1 != c and n2 != c:
            return 'Otra clase aislada 1 año'
        if n1 == c and n2 != c:
            return 'Otra clase aislada 2 años'
    return 'Otro patrón'


def codigo_clases(p1, c, n1, n2):
    p1, c, n1, n2 = map(int, (p1, c, n1, n2))
    return f'{p1}-{c}-{n1}-{n2}'


def codigo_desde_k(k):
    k = int(float(k))
    return codigo_clases(
        k // 1_000_000,
        (k % 1_000_000) // 10_000,
        (k % 10_000) // 100,
        k % 100,
    )


def parse_trajectory(code):
    k = int(float(code))
    p1, c, n1, n2 = k // 1_000_000, (k % 1_000_000) // 10_000, (k % 10_000) // 100, k % 100
    ids = codigo_clases(p1, c, n1, n2)
    return {
        'codigo': k,
        'clase_1': p1, 'clase_2': c, 'clase_3': n1, 'clase_4': n2,
        'clases': ids,
        'trayectoria': ids,
        'tipo': describe_trayectoria(p1, c, n1, n2),
    }


def folder_para_version(version, folder_ft=None, folder_class=None):
    """v1 → clasificación base; v>1 o None → filtros (clasificacion-ft)."""
    folder_ft = folder_ft or FOLDER_FT
    folder_class = folder_class or FOLDER_CLASS
    if version is not None and int(version) == 1:
        return folder_class
    return folder_ft


def geometria_region(region_id, vector_path=None):
    """Geometría ROI (id_regionC) desde el vector buffer C4/C5."""
    path = vector_path or REGION_VECTOR
    rid = int(str(region_id).strip())
    return (
        ee.FeatureCollection(path)
        .filter(ee.Filter.eq('id_regionC', rid))
        .geometry()
    )


def cargar_asset_region(region_id, folder=None, version=None):
    """Busca COLOMBIA-{region_id}-{version} en GEE.

    version=None → la más alta en filtros (excluye 11 y 99).
    version=1    → asset en clasificacion.
    version>1    → asset en clasificacion-ft.
    """
    rid = str(region_id).strip()
    prefix = f'COLOMBIA-{rid}-'
    parent = folder or folder_para_version(version)
    assets = ee.data.listAssets({'parent': parent})['assets']

    if version is not None:
        ver = str(int(version))
        target = f'{prefix}{ver}'
        for a in assets:
            if a['name'].split('/')[-1] == target:
                return ee.Image(a['name']), a['name']
        return None, None

    matches = []
    for a in assets:
        base = a['name'].split('/')[-1]
        if not base.startswith(prefix):
            continue
        ver = base[len(prefix):]
        if ver.isdigit() and int(ver) not in (11, 99):
            matches.append(a)
    matches = sorted(
        matches,
        key=lambda a: int(a['name'].split('/')[-1].rsplit('-', 1)[-1]),
        reverse=True,
    )
    if not matches:
        return None, None
    name = matches[0]['name']
    return ee.Image(name), name


def cargar_mosaico_regiones(region_ids, folder=FOLDER):
    """Mosaico con la versión más reciente de todas las regiones disponibles."""
    ids = {str(rid).strip() for rid in region_ids}
    disponibles = {}
    for asset in ee.data.listAssets({'parent': folder})['assets']:
        name = asset['name']
        base = name.split('/')[-1]
        if not base.startswith('COLOMBIA-'):
            continue
        body = base[len('COLOMBIA-'):]
        if '-' not in body:
            continue
        rid, version = body.rsplit('-', 1)
        if rid not in ids or not version.isdigit() or int(version) in (11, 99):
            continue
        disponibles.setdefault(rid, []).append((int(version), name))

    seleccionados = {
        rid: max(opciones, key=lambda item: item[0])[1]
        for rid, opciones in disponibles.items()
    }
    nombres = [seleccionados[str(rid)] for rid in region_ids if str(rid) in seleccionados]
    faltantes = [str(rid) for rid in region_ids if str(rid) not in seleccionados]
    bandas = [f'classification_{year}' for year in range(1985, 2026)]
    imagenes = [ee.Image(name).select(bandas).toByte() for name in nombres]
    mosaico = ee.ImageCollection.fromImages(imagenes).mosaic()
    return mosaico, nombres, faltantes


def histogramas_trayectorias_por_anio(imagen):
    """Histogramas de trayectorias por año en una sola reducción regional."""
    bandas = []
    for year in range(1986, 2024):
        prev1 = imagen.select(band_name(year - 1))
        curr = imagen.select(band_name(year))
        next1 = imagen.select(band_name(year + 1))
        next2 = imagen.select(band_name(year + 2))
        anomalia = anomalia_bosque(prev1, curr, next1, next2)
        trayectoria = prev1.multiply(1_000_000).add(curr.multiply(10_000)).add(next1.multiply(100)).add(next2)
        bandas.append(trayectoria.updateMask(anomalia).rename(f'trajectory_{year}'))

    hist = ee.Image.cat(bandas).reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=imagen.geometry(), scale=30, maxPixels=1e10, tileScale=8,
    ).getInfo()
    return {
        year: hist.get(f'trajectory_{year}') or {}
        for year in range(1986, 2024)
    }


def ids_desde_regiones(path=None):
    path = path or REGIONES
    """Lee IDs de región desde la primera columna del xlsx (sin encabezado), en orden del archivo."""
    col = pd.read_excel(path, header=None, usecols=[0]).iloc[:, 0].dropna()
    ids = col.astype(int).astype(str).drop_duplicates().tolist()
    if not ids:
        raise ValueError(f'No se encontraron IDs de región en {path}')
    return ids


def corregir_falsos_bosques(imagen, pasadas=2):
    """Corrige en bloque bosque aislado residual, incluido A-3-B."""

    def _paso(img):
        # Años centrales con ventana completa t-2 ... t+2.
        years = list(range(1987, 2024))
        curr_names = [f'classification_{y}' for y in years]

        def serie(offset):
            names = [f'classification_{y + offset}' for y in years]
            return img.select(names).rename(curr_names)

        prev2, prev1 = serie(-2), serie(-1)
        curr, next1, next2 = serie(0), serie(1), serie(2)

        error1 = prev1.neq(CLASE_BOSQUE).And(curr.eq(CLASE_BOSQUE)).And(next1.neq(CLASE_BOSQUE))
        error2_ini = prev1.neq(CLASE_BOSQUE).And(curr.eq(CLASE_BOSQUE)).And(next1.eq(CLASE_BOSQUE)).And(next2.neq(CLASE_BOSQUE))
        error2_fin = prev2.neq(CLASE_BOSQUE).And(prev1.eq(CLASE_BOSQUE)).And(curr.eq(CLASE_BOSQUE)).And(next1.neq(CLASE_BOSQUE))

        rep1 = reemplazo_falso_bosque(prev1, next1)
        rep2_ini = reemplazo_falso_bosque_2anio_ini(prev1, next1, next2)
        rep2_fin = reemplazo_falso_bosque(prev2, next1)
        corrected = curr.where(error1, rep1).where(error2_ini, rep2_ini).where(error2_fin, rep2_fin)
        out = img.addBands(corrected, curr_names, True)

        # Bordes sin ventana de cinco años: solo A-3-B.
        for y in (1986, 2024):
            curr_edge = out.select(band_name(y))
            prev_edge = out.select(band_name(y - 1))
            next_edge = out.select(band_name(y + 1))
            error = prev_edge.neq(CLASE_BOSQUE).And(curr_edge.eq(CLASE_BOSQUE)).And(next_edge.neq(CLASE_BOSQUE))
            fixed = curr_edge.where(error, reemplazo_falso_bosque(prev_edge, next_edge))
            out = out.addBands(fixed, [f'classification_{y}'], True)
        return out.toByte()

    result = imagen
    for _ in range(pasadas):
        result = _paso(result)
    return result


def _rellenar_borde_temporal(imagen, valor, year_min=1985, year_max=2025):
    """Extiende una clase estable de dos años hacia el primer/último año."""
    first = imagen.select(band_name(year_min))
    second = imagen.select(band_name(year_min + 1))
    third = imagen.select(band_name(year_min + 2))
    mask_first = first.neq(valor).And(second.eq(valor)).And(third.eq(valor))
    imagen = imagen.addBands(first.where(mask_first, valor), [f'classification_{year_min}'], True)

    before2 = imagen.select(band_name(year_max - 2))
    before1 = imagen.select(band_name(year_max - 1))
    last = imagen.select(band_name(year_max))
    mask_last = before2.eq(valor).And(before1.eq(valor)).And(last.neq(valor))
    return imagen.addBands(last.where(mask_last, valor), [f'classification_{year_max}'], True)


def _rellenar_hueco(imagen, valor, longitud, year_min=1985, year_max=2025):
    """Rellena A-X...-A; longitud 1, 2 o 3 años internos."""
    starts = list(range(year_min + 1, year_max - longitud + 1))
    prev_names = [f'classification_{y - 1}' for y in starts]
    after_names = [f'classification_{y + longitud}' for y in starts]
    prev = imagen.select(prev_names)
    after = imagen.select(after_names)
    mask = prev.eq(valor).And(after.eq(valor))

    for offset in range(longitud):
        names = [f'classification_{y + offset}' for y in starts]
        inside = imagen.select(names)
        mask = mask.And(inside.neq(valor))

    for offset in range(longitud):
        names = [f'classification_{y + offset}' for y in starts]
        inside = imagen.select(names)
        corrected = inside.where(mask.rename(inside.bandNames()), valor)
        imagen = imagen.addBands(corrected, names, True)
    return imagen


def corregir_plantaciones_aisladas(imagen, pasadas=2):
    """Plantación 9/35/74 → bosque cuando:

    - está aislada entre bosque (1–2 años), o
    - dura 1–2 años y después viene bosque (p. ej. 21-35-3-3-3 → 21-3-3-3-3).

    No corrige plantaciones de 3+ años seguidas.
    """

    def _paso(img):
        years = list(range(1987, 2024))
        curr_names = [f'classification_{y}' for y in years]

        def serie(offset):
            names = [f'classification_{y + offset}' for y in years]
            return img.select(names).rename(curr_names)

        prev2, prev1 = serie(-2), serie(-1)
        curr, next1, next2 = serie(0), serie(1), serie(2)

        # Entre bosque
        entre_1 = prev1.eq(CLASE_BOSQUE).And(es_plantacion(curr)).And(next1.eq(CLASE_BOSQUE))
        entre_2_ini = (
            prev1.eq(CLASE_BOSQUE)
            .And(es_plantacion(curr))
            .And(es_plantacion(next1))
            .And(next2.eq(CLASE_BOSQUE))
        )
        entre_2_fin = (
            prev2.eq(CLASE_BOSQUE)
            .And(es_plantacion(prev1))
            .And(es_plantacion(curr))
            .And(next1.eq(CLASE_BOSQUE))
        )

        # 1–2 años de plantación seguidos de bosque
        antes_1 = es_plantacion(prev1).Not().And(es_plantacion(curr)).And(next1.eq(CLASE_BOSQUE))
        antes_2_ini = (
            es_plantacion(prev1).Not()
            .And(es_plantacion(curr))
            .And(es_plantacion(next1))
            .And(next2.eq(CLASE_BOSQUE))
        )
        antes_2_fin = (
            es_plantacion(prev2).Not()
            .And(es_plantacion(prev1))
            .And(es_plantacion(curr))
            .And(next1.eq(CLASE_BOSQUE))
        )

        mask = (
            entre_1.Or(entre_2_ini).Or(entre_2_fin)
            .Or(antes_1).Or(antes_2_ini).Or(antes_2_fin)
        )
        corrected = curr.where(mask, CLASE_BOSQUE)
        out = img.addBands(corrected, curr_names, True)

        for y in (1986, 2024):
            curr_edge = out.select(band_name(y))
            prev_edge = out.select(band_name(y - 1))
            next_edge = out.select(band_name(y + 1))
            error = (
                (
                    prev_edge.eq(CLASE_BOSQUE)
                    .And(es_plantacion(curr_edge))
                    .And(next_edge.eq(CLASE_BOSQUE))
                ).Or(
                    es_plantacion(prev_edge).Not()
                    .And(es_plantacion(curr_edge))
                    .And(next_edge.eq(CLASE_BOSQUE))
                )
            )
            fixed = curr_edge.where(error, CLASE_BOSQUE)
            out = out.addBands(fixed, [f'classification_{y}'], True)
        return out.toByte()

    result = imagen
    for _ in range(pasadas):
        result = _paso(result)
    return result


def corregir_bosque_y_plantaciones(imagen, pasadas=2):
    """Alterna falsos bosques y plantaciones cortas (entre bosque o antes de bosque)."""
    result = imagen
    for _ in range(pasadas):
        result = corregir_falsos_bosques(result, pasadas=1)
        result = corregir_plantaciones_aisladas(result, pasadas=1)
    return result


def corregir_matriz_importancia(
    imagen,
    prioridad=PRIORIDAD_CLASES,
    ventanas=VENTANAS_HUECOS,
    corregir_bordes=CORREGIR_BORDES,
    pasadas_bosque=PASADAS_BOSQUE_RESIDUAL,
    modo=None,
):
    modo = (modo or MODO_CORRECCION or 'todas').strip().lower()
    if modo not in ('todas', 'bosque'):
        raise ValueError("MODO_CORRECCION debe ser 'todas' o 'bosque'")

    if modo == 'bosque':
        result = corregir_bosque_y_plantaciones(imagen, pasadas=pasadas_bosque)
        return result.select(imagen.bandNames()).toByte()

    result = imagen.toByte()
    orden_ejecucion = list(reversed(tuple(prioridad)))

    if corregir_bordes:
        for valor in orden_ejecucion:
            result = _rellenar_borde_temporal(result, valor)

    if 3 in ventanas:
        for valor in orden_ejecucion:
            result = _rellenar_hueco(result, valor, longitud=1)
    for ventana in (4, 5):
        if ventana in ventanas:
            for valor in orden_ejecucion:
                result = _rellenar_hueco(result, valor, longitud=ventana - 2)
    if 3 in ventanas:
        for valor in orden_ejecucion:
            result = _rellenar_hueco(result, valor, longitud=1)

    result = corregir_bosque_y_plantaciones(result, pasadas=pasadas_bosque)
    return result.select(imagen.bandNames()).toByte()


## 4. Carga de datos


In [ ]:

if not REGIONES_XLSX.exists():
    raise FileNotFoundError(f'Falta {REGIONES_XLSX} (carpeta data/)')

REGION_IDS = ids_desde_regiones(REGIONES_XLSX)
MODO_MAPA = REGION_ID is not None


def _msg_correccion():
    modo = str(MODO_CORRECCION).strip().lower()
    plant = '/'.join(str(c) for c in CLASES_PLANTACION)
    if modo == 'bosque':
        return (
            f"Corrección: bosque↔plantación ({plant}) "
            f"· {PASADAS_BOSQUE_RESIDUAL} pasadas"
        )
    return (
        f"Corrección: matriz ATBD ({len(PRIORIDAD_CLASES)} clases) "
        f"+ residual bosque↔plantación ({plant}) · modo '{modo}'"
    )


print(f'Leyenda: {len(CLASS_NAMES)} clases desde {_leyenda["source"]}')
print(f'Regiones en {REGIONES_XLSX.name}: {len(REGION_IDS)}')
print(f'Modo corrección: {MODO_CORRECCION}')

if MODO_MAPA:
    rid = str(REGION_ID)
    if rid not in REGION_IDS:
        print(f'Aviso: {rid} no aparece en {REGIONES_XLSX.name}')
    IMG_ORIGINAL, ASSET_ORIGINAL = cargar_asset_region(rid, version=VERSION_INPUT)
    if IMG_ORIGINAL is None:
        msg = (
            f'No hay asset LULC para la región {rid}'
            + (f' versión {VERSION_INPUT}' if VERSION_INPUT is not None else '')
        )
        raise ValueError(msg)
    REGION_GEO = geometria_region(rid)
    IMG_CORREGIDA = corregir_matriz_importancia(IMG_ORIGINAL)
    print(f'Región cargada: {rid} · versión: {VERSION_INPUT if VERSION_INPUT is not None else "auto (máx)"}')
    print(f'Asset: {ASSET_ORIGINAL}')
    print(f'ROI: {REGION_VECTOR.split("/")[-1]} · id_regionC={rid}')
    print(_msg_correccion())
else:
    REGION_GEO = None
    IMG_ORIGINAL, ASSETS_MOSAICO, REGIONES_FALTANTES = cargar_mosaico_regiones(REGION_IDS)
    IMG_CORREGIDA = corregir_matriz_importancia(IMG_ORIGINAL)
    ASSET_ORIGINAL = 'MOSAICO_REGIONES'
    print(f'Mosaico nacional cargado: {len(ASSETS_MOSAICO)}/{len(REGION_IDS)} regiones con asset')
    print(_msg_correccion())
    if REGIONES_FALTANTES:
        print(f'Regiones sin asset: {len(REGIONES_FALTANTES)}')


## 5. Exportar por regiones


In [ ]:
if MODO_MAPA:
    print('Modo mapa activo: export batch omitido.')
elif not REEXPORTAR and OUTPUT_CSV.exists():
    print(f'CSV ya existe → no se reexporta: {OUTPUT_CSV.resolve()}')
    print('Para regenerarlo, pon REEXPORTAR = True en Configuración y vuelve a ejecutar.')
    df_tray = pd.read_csv(OUTPUT_CSV, encoding='utf-8-sig')
    print(f'Filas: {len(df_tray):,} · Regiones: {df_tray["region_id"].nunique()}')
    display(df_tray.head(10))
elif not REEXPORTAR and not OUTPUT_CSV.exists():
    raise FileNotFoundError(
        f'No existe {OUTPUT_CSV}. Pon REEXPORTAR = True para generarlo, '
        'o coloca el CSV en la carpeta del proyecto.'
    )
else:
    region_ids = REGION_IDS
    print(f'Reexportando {len(region_ids)} regiones → {OUTPUT_CSV.name}')

    filas = []
    omitidas = []
    sin_anomalias = []

    for i, rid in enumerate(region_ids, 1):
        print(f'[{i}/{len(region_ids)}] {rid}...', end=' ')
        try:
            img, asset = cargar_asset_region(rid)
        except Exception as err:
            print(f'error ({err})')
            omitidas.append({'region_id': rid, 'motivo': str(err)})
            continue
        if img is None:
            print('sin asset')
            omitidas.append({'region_id': rid, 'motivo': 'sin asset'})
            continue
        try:
            hist_por_anio = histogramas_trayectorias_por_anio(img)
        except Exception as err:
            print(f'error GEE ({err})')
            omitidas.append({'region_id': rid, 'motivo': str(err), 'asset': asset})
            continue
        if not any(hist_por_anio.values()):
            print('sin anomalias')
            sin_anomalias.append(rid)
            continue
        n_patrones = 0
        for year, hist in hist_por_anio.items():
            for codigo, pixeles in hist.items():
                filas.append({
                    'region_id': rid,
                    'year': year,
                    'asset': asset,
                    **parse_trajectory(codigo),
                    'pixeles': int(pixeles),
                })
                n_patrones += 1
        print(f'{n_patrones} patrones-año')

    df_tray = pd.DataFrame(filas).sort_values(
        ['region_id', 'year', 'pixeles'], ascending=[True, True, False]
    )
    df_tray.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
    print(f'\nGuardado: {OUTPUT_CSV.resolve()}')
    n_reg = df_tray['region_id'].nunique() if len(df_tray) else 0
    print(f'Filas: {len(df_tray):,} · Regiones con datos: {n_reg}')
    print(f'Omitidas: {len(omitidas)} · Sin anomalias: {len(sin_anomalias)}')
    if omitidas:
        display(pd.DataFrame(omitidas).head(10))
    display(df_tray.head(10))


## 6. Análisis nacional o regional


In [ ]:
from importlib import reload
import matplotlib.pyplot as plt
import analisis_nacional

reload(analisis_nacional)
plt.style.use('default')
RESULTADOS = analisis_nacional.analizar(
    region_id=REGION_ID,
    total_colombia=TOTAL_PIXELES_COLOMBIA_30M,
)
print('Listo. Claves:', list(RESULTADOS.keys()))


## 6b. Final de la serie (2023–2025)


In [ ]:
from importlib import reload
import analisis_extremos

reload(analisis_extremos)
EXTREMOS = analisis_extremos.analizar_extremos()
print('Claves:', list(EXTREMOS.keys()))
EXTREMOS['resumen_2023']

## 7. Estadísticas de coberturas


In [ ]:
from importlib import reload

import plotly.io as pio
import estadisticas_correccion

reload(estadisticas_correccion)
pio.renderers.default = "vscode"

STATS, FIG = estadisticas_correccion.calcular_y_mostrar(
    IMG_ORIGINAL,
    IMG_CORREGIDA,
    region_id=REGION_ID,
    geometry=REGION_GEO,
    class_names=CLASS_NAMES,
    year_min=1985,
    year_max=2026,
)


## 8. Mapa interactivo


In [11]:
if IMG_ORIGINAL is None:
    from IPython.display import HTML, clear_output, display
    clear_output(wait=True)
    display(HTML(
        '<p style="font-family:system-ui;color:#666">'
        'Ejecuta primero la celda de carga de datos.</p>'
    ))
else:
    import io
    import time
    import ipywidgets as widgets
    import leafmap
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from IPython.display import clear_output, display

    plt.style.use('default')
    plt.rcParams.update({
        'text.color': '#1a202c',
        'axes.labelcolor': '#1a202c',
        'axes.titlecolor': '#1a202c',
        'xtick.color': '#4a5568',
        'ytick.color': '#4a5568',
        'legend.labelcolor': '#1a202c',
        'figure.facecolor': 'white',
        'axes.facecolor': 'white',
        'savefig.facecolor': 'white',
    })

    if '_MAP_UI' not in globals():
        _MAP_UI = {'seq': 0, 'busy': False}
    _MAP_UI['seq'] += 1
    _MAP_UI['busy'] = False
    MAP_SEQ = _MAP_UI['seq']

    modo_mapa = str(globals().get('MAPA_ANOMALIAS', 'bosque')).strip().lower()
    if modo_mapa in ('todas', 'all', 'bosque_plantacion_otras') or 'otra' in modo_mapa:
        modo_mapa = 'todas'
    elif modo_mapa in ('bosque_plantacion', 'bosque+plantacion', 'plantacion', 'plantaciones') or 'plant' in modo_mapa:
        modo_mapa = 'bosque_plantacion'
    else:
        modo_mapa = 'bosque'

    state = {
        'map_seq': MAP_SEQ,
        'pixel': None,
        'marker': None,
        'last_click_ts': 0.0,
    }

    chart_title = widgets.HTML(
        value="<div style='color:#718096;font-style:italic;padding:36px 12px;text-align:center'>"
              "Clic en el mapa para ver la serie del píxel.</div>"
    )
    chart_img = widgets.Image(format='png', layout=widgets.Layout(width='100%', display='none'))
    chart_box = widgets.VBox(
        [chart_title, chart_img],
        layout=widgets.Layout(
            width='100%', min_height='420px', padding='10px',
            border='1px solid #cbd5e0', border_radius='10px',
        ),
    )
    table_html = widgets.HTML(
        value=(
            "<div style='color:#718096;padding:16px'>"
            "Pulse <b>Calcular tabla</b> para obtener el histograma de patrones "
            "(consulta a Earth Engine).</div>"
        ),
        layout=widgets.Layout(
            width='100%', min_height='240px', max_height='480px', overflow_y='auto',
            padding='8px', border='1px solid #cbd5e0', border_radius='10px',
        ),
    )
    btn_tablas = widgets.Button(
        description='Calcular tabla de patrones',
        button_style='info',
        layout=widgets.Layout(width='240px', height='36px'),
    )
    status_html = widgets.HTML(value='')

    def _section_title(title, subtitle=''):
        sub = f'<div style="font-size:12px;color:#718096;margin-top:4px">{subtitle}</div>' if subtitle else ''
        return widgets.HTML(
            f'<div style="font-family:system-ui;width:100%;border-bottom:2px solid #2b6cb0;'
            f'padding:10px 0 8px 0;margin:16px 0 10px 0">'
            f'<div style="font-size:17px;font-weight:700;color:#1a365d">{title}</div>{sub}</div>'
        )

    MAP_H = '560px'
    m_orig = leafmap.Map(center=[4.5, -73.0], zoom=6, height=MAP_H)
    m_orig.layout = widgets.Layout(width='100%', height=MAP_H)
    m_orig.add_basemap('CartoDB.DarkMatter')

    TABLE_CSS = """
    <style>
    .traj-wrap { font-family: system-ui, sans-serif; }
    .traj-wrap h3 { margin: 0 0 10px 0; font-size: 15px; color: #1a202c; }
    .traj-wrap .meta { color: #718096; font-size: 12px; margin-bottom: 8px; }
    .traj-table { width: 100%; border-collapse: collapse; font-size: 12px; }
    .traj-table th {
        background: #2d3748; color: #fff; padding: 8px; text-align: left;
        position: sticky; top: 0; z-index: 1;
    }
    .traj-table td { padding: 7px 8px; border-bottom: 1px solid #e2e8f0; }
    .traj-table td.traj { max-width: 420px; word-wrap: break-word; }
    .traj-table tr:nth-child(even) { background: #f8fafc; }
    .traj-table .num { text-align: right; font-variant-numeric: tabular-nums; }
    .empty { color: #718096; font-style: italic; padding: 20px 8px; }
    </style>
    """

    def mostrar_placeholder_pixel(msg='Clic en el mapa para ver la serie del píxel.'):
        chart_img.layout.display = 'none'
        chart_img.value = b''
        chart_title.value = (
            f"<div style='color:#718096;font-style:italic;padding:36px 12px;text-align:center'>{msg}</div>"
        )

    def mostrar_pixel_cargando(lat, lon, texto='Consultando serie…'):
        chart_img.layout.display = 'none'
        chart_img.value = b''
        chart_title.value = (
            f"<div style='font-family:system-ui;color:#1a202c;padding:8px 4px'>"
            f"<b>Píxel</b> · {lat:.4f}, {lon:.4f}<br>"
            f"<span style='color:#718096;font-size:12px'>{texto}</span></div>"
        )

    def mostrar_pixel_resultado(lat, lon, png_bytes=None, empty=False, error=None, n_cambios=None):
        if error:
            chart_img.layout.display = 'none'
            chart_img.value = b''
            chart_title.value = (
                f"<div style='font-family:system-ui;color:#c53030;padding:12px'>"
                f"Error en píxel {lat:.4f}, {lon:.4f}: {error}</div>"
            )
            return
        if empty or not png_bytes:
            chart_img.layout.display = 'none'
            chart_img.value = b''
            chart_title.value = (
                f"<div style='font-family:system-ui;color:#718096;padding:12px'>"
                f"Píxel {lat:.4f}, {lon:.4f}: sin datos.</div>"
            )
            return
        if n_cambios == 0:
            extra = " · <span style='color:#2f855a'>sin cambios</span>"
        elif n_cambios is not None:
            extra = f" · <span style='color:#c05621'><b>{n_cambios} año(s) cambiaron</b></span>"
        else:
            extra = ''
        chart_title.value = (
            f"<div style='font-family:system-ui;color:#1a202c;padding:4px 0 8px 0'>"
            f"<b>Píxel</b> · {lat:.4f}, {lon:.4f}{extra}<br>"
            f"<span style='color:#718096;font-size:12px'>"
            f"Arriba original · Abajo corregida · Rojo = años distintos</span></div>"
        )
        chart_img.value = png_bytes
        chart_img.layout.display = 'block'

    def html_tabla_trayectorias(df, titulo):
        total_pix = int(df['pixeles'].sum())
        show = df.head(15).copy()
        rows = []
        for _, r in show.iterrows():
            rows.append(
                f"<tr><td class='traj'>{r['trayectoria']}</td>"
                f"<td>{r['tipo']}</td>"
                f"<td class='num'>{int(r['pixeles']):,}</td></tr>"
            )
        body = ''.join(rows) if rows else "<tr><td colspan='3' class='empty'>Sin anomalías.</td></tr>"
        return (
            f"{TABLE_CSS}<div class='traj-wrap'>"
            f"<h3>{titulo}</h3>"
            f"<div class='meta'>{len(df):,} patrones · {total_pix:,} píxeles · top 15</div>"
            f"<table class='traj-table'><thead><tr>"
            f"<th>Trayectoria</th><th>Tipo</th><th>Píxeles</th>"
            f"</tr></thead><tbody>{body}</tbody></table></div>"
        )

    def props_a_serie(properties):
        df = pd.DataFrame(
            [(int(k), int(v)) for k, v in properties.items() if str(k).isdigit() and v is not None],
            columns=['Año', 'Clase'],
        ).sort_values('Año')
        return df

    def sample_pixel_serie(img, lon, lat):
        point = ee.Geometry.Point([lon, lat])
        bands = img.bandNames()
        years = bands.map(lambda b: ee.String(ee.List(ee.String(b).split('_')).get(1)))
        return img.select(bands, years).reduceRegion(
            reducer=ee.Reducer.first(), geometry=point, scale=30,
        ).getInfo()

    def _dibujar_serie(ax, df, titulo, highlight_years=None):
        highlight_years = set(highlight_years or [])
        colors = df['Clase'].map(lulc_color)
        ax.set_facecolor('#f7fafc')
        ax.axhspan(CLASE_BOSQUE - 0.4, CLASE_BOSQUE + 0.4, color='#1f8d49', alpha=0.12)
        ax.plot(df['Año'], df['Clase'], color='#a0aec0', linestyle='-', alpha=0.7, zorder=1)
        ax.scatter(df['Año'], df['Clase'], c=colors, s=70, edgecolors='#2d3748', linewidths=0.5, zorder=2)
        if highlight_years:
            mask = df['Año'].isin(highlight_years)
            ax.scatter(
                df.loc[mask, 'Año'], df.loc[mask, 'Clase'],
                s=160, facecolors='none', edgecolors='#c53030', linewidths=2.2, zorder=3,
            )
        ax.set_ylim(0, min(len(PALETTE_LULC) - 1, max(40, int(df['Clase'].max()) + 2)))
        ax.set_title(titulo, fontsize=11, color='#1a202c', loc='left')
        ax.set_ylabel('Clase LULC', color='#1a202c')
        ax.tick_params(colors='#4a5568', labelsize=8)
        ax.grid(axis='y', color='#e2e8f0')
        return sorted(df['Clase'].unique())

    def render_compare_png(lat, lon, df_orig, df_corr):
        if df_orig.empty and df_corr.empty:
            return None, 0
        merged = df_orig.merge(df_corr, on='Año', how='outer', suffixes=('_orig', '_corr')).sort_values('Año')
        cambiados = merged[
            merged['Clase_orig'].notna() & merged['Clase_corr'].notna()
            & (merged['Clase_orig'] != merged['Clase_corr'])
        ]['Año'].astype(int).tolist()
        fig, axes = plt.subplots(2, 1, figsize=(12, 6.0), sharex=True, facecolor='white')
        clases = set()
        if not df_orig.empty:
            clases.update(_dibujar_serie(axes[0], df_orig, 'Original', highlight_years=cambiados))
        else:
            axes[0].text(0.5, 0.5, 'Sin datos', ha='center', va='center', transform=axes[0].transAxes)
        if not df_corr.empty:
            clases.update(_dibujar_serie(axes[1], df_corr, 'Corregida', highlight_years=cambiados))
        else:
            axes[1].text(0.5, 0.5, 'Sin datos', ha='center', va='center', transform=axes[1].transAxes)
        axes[1].set_xlabel('Año', color='#1a202c')
        years = sorted(set(df_orig['Año']).union(df_corr['Año']))
        if years:
            step = 2 if len(years) > 20 else 1
            axes[1].set_xticks(years[::step])
            axes[1].set_xticklabels(years[::step], rotation=45, ha='right')
        handles = [
            plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=lulc_color(c),
                       markeredgecolor='#2d3748', markersize=8, label=f'{c} — {class_name(c)}')
            for c in sorted(clases)
        ]
        if cambiados:
            handles.append(plt.Line2D(
                [0], [0], marker='o', color='w', markerfacecolor='none',
                markeredgecolor='#c53030', markersize=10, label='Año que cambió',
            ))
        leg = fig.legend(
            handles=handles, loc='lower center', ncol=min(4, max(1, len(handles))),
            fontsize=8, framealpha=0.95, title='Clases LULC',
            bbox_to_anchor=(0.5, -0.02), facecolor='white', edgecolor='#cbd5e0',
            labelcolor='#1a202c',
        )
        if leg is not None:
            for t in leg.get_texts():
                t.set_color('#1a202c')
            if leg.get_title() is not None:
                leg.get_title().set_color('#1a202c')
        fig.suptitle(f'Antes vs después · {lat:.4f}, {lon:.4f}', fontsize=12, color='#1a202c', y=1.01)
        fig.tight_layout()
        fig.subplots_adjust(bottom=0.16, hspace=0.28)
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=100, bbox_inches='tight', facecolor='white')
        plt.close(fig)
        return buf.getvalue(), len(cambiados)

    def refrescar_serie_pixel(lat, lon):
        # Muestra original vs corregida (punto). No usa el mapa corregido.
        mostrar_pixel_cargando(lat, lon, 'Consultando original y corregida…')
        props_o = sample_pixel_serie(IMG_ORIGINAL, lon, lat) or {}
        props_c = sample_pixel_serie(IMG_CORREGIDA, lon, lat) or {}
        df_o = props_a_serie(props_o)
        df_c = props_a_serie(props_c)
        if df_o.empty and df_c.empty:
            mostrar_pixel_resultado(lat, lon, empty=True)
            return
        png, n_cambios = render_compare_png(lat, lon, df_o, df_c)
        mostrar_pixel_resultado(lat, lon, png_bytes=png, empty=png is None, n_cambios=n_cambios)

    def poner_marcador(lat, lon):
        marker = state.get('marker')
        if marker is not None:
            try:
                marker.location = (lat, lon)
                return
            except Exception:
                try:
                    m_orig.remove_layer(marker)
                except Exception:
                    pass
                state['marker'] = None
        m_orig.add_marker(location=[lat, lon], name='px_sel', draggable=False)
        state['marker'] = m_orig.find_layer('px_sel')

    def _roi_mapa():
        geo = globals().get('REGION_GEO')
        if geo is not None:
            return geo
        if REGION_ID is not None:
            try:
                return geometria_region(REGION_ID)
            except Exception:
                pass
        return IMG_ORIGINAL.geometry()

    YEAR_MIN, YEAR_MAX = 1986, 2024
    COLORES_VENTANA = {
        3: '#f46d43',
        4: '#228be6',
        5: '#9f7aea',
    }
    LEGEND_VENTANA = {
        'Ventana 3 (A–X–A)': 'f46d43',
        'Ventana 4 (A–XX–A)': '228be6',
        'Ventana 5 (A–XXX–A)': '9f7aea',
    }
    LAYER_NAMES = {v: f'ventana_{v}' for v in (3, 4, 5)}

    year_selector = widgets.BoundedIntText(
        value=min(2023, YEAR_MAX),
        min=YEAR_MIN,
        max=YEAR_MAX,
        step=1,
        description='Año:',
        style={'description_width': '40px'},
        layout=widgets.Layout(width='160px'),
    )
    btn_actualizar = widgets.Button(
        description='Actualizar mapa',
        button_style='primary',
        layout=widgets.Layout(width='150px', height='36px'),
    )
    controles = widgets.HBox(
        [year_selector, btn_actualizar],
        layout=widgets.Layout(gap='10px', align_items='center', margin='4px 0 8px 0'),
    )

    def _band(img, year):
        return img.select(f'classification_{int(year)}')

    def _imagen_ventana(imagen, year):
        need = []
        for d in range(-4, 5):
            yy = int(year) + d
            if 1985 <= yy <= 2025:
                need.append(f'classification_{yy}')
        return imagen.select(sorted(set(need)))

    def _mascara_isla_bosque(img, year, longitud):
        masks = []
        for pos in range(longitud):
            y0 = int(year) - pos
            yp, ya = y0 - 1, y0 + longitud
            if yp < 1985 or ya > 2025:
                continue
            m = _band(img, yp).neq(CLASE_BOSQUE).And(_band(img, ya).neq(CLASE_BOSQUE))
            for k in range(longitud):
                m = m.And(_band(img, y0 + k).eq(CLASE_BOSQUE))
            masks.append(m.unmask(0))
        if not masks:
            return ee.Image(0)
        out = masks[0]
        for m in masks[1:]:
            out = out.Or(m)
        return out

    def _mascara_isla_plantacion(img, year, longitud):
        masks = []
        for pos in range(longitud):
            y0 = int(year) - pos
            yp, ya = y0 - 1, y0 + longitud
            if yp < 1985 or ya > 2025:
                continue
            prev = _band(img, yp)
            after = _band(img, ya)
            plants = es_plantacion(_band(img, y0))
            for k in range(1, longitud):
                plants = plants.And(es_plantacion(_band(img, y0 + k)))
            entre = prev.eq(CLASE_BOSQUE).And(plants).And(after.eq(CLASE_BOSQUE))
            antes = es_plantacion(prev).Not().And(plants).And(after.eq(CLASE_BOSQUE))
            masks.append(entre.Or(antes).unmask(0))
        if not masks:
            return ee.Image(0)
        out = masks[0]
        for m in masks[1:]:
            out = out.Or(m)
        return out

    def _mascara_isla_otras(img, year, longitud):
        masks = []
        for pos in range(longitud):
            y0 = int(year) - pos
            yp, ya = y0 - 1, y0 + longitud
            if yp < 1985 or ya > 2025:
                continue
            curr = _band(img, y0)
            m = es_otra_clase(curr).And(_band(img, yp).neq(curr)).And(_band(img, ya).neq(curr))
            for k in range(1, longitud):
                m = m.And(_band(img, y0 + k).eq(curr))
            masks.append(m.unmask(0))
        if not masks:
            return ee.Image(0)
        out = masks[0]
        for m in masks[1:]:
            out = out.Or(m)
        return out

    def construir_capa_ventana(imagen, modo, year, ventana, roi=None):
        """Anomalías del año en una ventana (3=A–X–A, 4=A–XX–A, 5=A–XXX–A)."""
        longitud = int(ventana) - 2
        img = _imagen_ventana(imagen, year)
        mask = ee.Image(0)
        if modo == 'todas':
            mask = mask.Or(_mascara_isla_otras(img, year, longitud))
        if modo in ('bosque_plantacion', 'todas'):
            mask = mask.Or(_mascara_isla_plantacion(img, year, longitud))
        mask = mask.Or(_mascara_isla_bosque(img, year, longitud))
        out = mask.selfMask().rename(f'v{ventana}')
        if roi is not None:
            out = out.clip(roi)
        return out

    def construir_trayectorias_anio(imagen, modo, year, roi=None):
        """Histograma de patrones del año (ventanas 3 y 4 con tupla de 4 bandas)."""
        img = _imagen_ventana(imagen, year)
        y = int(year)
        prev1 = _band(img, y - 1)
        curr = _band(img, y)
        next1 = _band(img, y + 1)
        next2 = _band(img, min(y + 2, 2025))
        anom = ee.Image(0)
        for ventana in (3, 4, 5):
            anom = anom.Or(
                construir_capa_ventana(imagen, modo, year, ventana, roi=None).unmask(0).gt(0)
            )
        traj = prev1.multiply(1_000_000).add(curr.multiply(10_000)).add(next1.multiply(100)).add(next2)
        traj = traj.updateMask(anom).rename('trajectory')
        if roi is not None:
            traj = traj.clip(roi)
        return traj

    def _quitar_capas_ventana(mapa):
        for name in LAYER_NAMES.values():
            try:
                layer = mapa.find_layer(name)
            except Exception:
                layer = None
            if layer is not None:
                try:
                    mapa.remove_layer(layer)
                except Exception:
                    pass

    def agregar_capas_anio(mapa, imagen, modo, year, roi):
        _quitar_capas_ventana(mapa)
        for ventana in (3, 4, 5):
            capa = construir_capa_ventana(imagen, modo, year, ventana, roi=roi)
            vis = {
                'min': 1, 'max': 1,
                'palette': [COLORES_VENTANA[ventana]],
            }
            mapa.add_ee_layer(
                capa, vis, name=LAYER_NAMES[ventana], shown=True, opacity=0.85,
            )

    def cargar_tabla_patrones(traj_img, widget_html, titulo, roi):
        widget_html.value = (
            f"{TABLE_CSS}<div class='traj-wrap'><h3>{titulo}</h3>"
            f"<div class='empty'>Consultando Earth Engine…</div></div>"
        )
        hist = traj_img.reduceRegion(
            reducer=ee.Reducer.frequencyHistogram(),
            geometry=roi, scale=30, maxPixels=1e10, bestEffort=True, tileScale=4,
        ).getInfo()
        if not hist.get('trajectory'):
            widget_html.value = (
                f"{TABLE_CSS}<div class='traj-wrap'><h3>{titulo}</h3>"
                f"<div class='empty'>Sin trayectorias anómalas en la región.</div></div>"
            )
            return
        df = pd.DataFrame([
            {**parse_trajectory(k), 'pixeles': v} for k, v in hist['trajectory'].items()
        ]).sort_values('pixeles', ascending=False)
        widget_html.value = html_tabla_trayectorias(df, titulo)

    def on_cargar_tablas(_btn=None):
        btn_tablas.disabled = True
        year = int(year_selector.value)
        status_html.value = (
            "<div style='font-family:system-ui;color:#718096;font-size:12px;padding:4px 0'>"
            f"Consultando histogramas {year} en Earth Engine…</div>"
        )
        try:
            roi = _roi_mapa()
            traj = construir_trayectorias_anio(IMG_ORIGINAL, modo_mapa, year, roi=roi)
            cargar_tabla_patrones(
                traj, table_html, f'Patrones · {year} · clasificación original', roi,
            )
            status_html.value = (
                "<div style='font-family:system-ui;color:#2f855a;font-size:12px;padding:4px 0'>"
                "Tabla de patrones lista.</div>"
            )
        except Exception as err:
            status_html.value = (
                f"<div style='font-family:system-ui;color:#c53030;font-size:12px;padding:4px 0'>"
                f"Error al calcular la tabla: {err}</div>"
            )
        finally:
            btn_tablas.disabled = False

    def on_map_click(**kwargs):
        if state.get('map_seq') != _MAP_UI.get('seq'):
            return
        if kwargs.get('type') != 'click':
            return
        if _MAP_UI.get('busy'):
            return
        now = time.time()
        if now - state['last_click_ts'] < 0.5:
            return
        state['last_click_ts'] = now
        _MAP_UI['busy'] = True
        try:
            lat, lon = kwargs['coordinates']
            state['pixel'] = (lat, lon)
            poner_marcador(lat, lon)
            refrescar_serie_pixel(lat, lon)
        except Exception as err:
            coords = kwargs.get('coordinates', (0, 0))
            lat, lon = coords if isinstance(coords, (list, tuple)) else (0, 0)
            mostrar_pixel_resultado(lat, lon, error=err)
        finally:
            _MAP_UI['busy'] = False

    def refrescar_capas_anio(_btn=None):
        year = int(year_selector.value)
        btn_actualizar.disabled = True
        year_selector.disabled = True
        status_html.value = (
            "<div style='font-family:system-ui;color:#718096;font-size:12px;padding:4px 0'>"
            f"Cargando anomalías de {year} (ventanas 3, 4 y 5)…</div>"
        )
        try:
            roi = _roi_mapa()
            agregar_capas_anio(m_orig, IMG_ORIGINAL, modo_mapa, year, roi)
            status_html.value = (
                "<div style='font-family:system-ui;color:#2f855a;font-size:12px;padding:4px 0'>"
                f"Año <b>{year}</b> · naranja: ventana 3 · azul: ventana 4 · violeta: ventana 5. "
                "Active/desactive capas en el control del mapa.</div>"
            )
        except Exception as err:
            status_html.value = (
                f"<div style='font-family:system-ui;color:#c53030;font-size:12px;padding:4px 0'>"
                f"Error al cargar capas: {err}</div>"
            )
        finally:
            btn_actualizar.disabled = False
            year_selector.disabled = False

    m_orig.on_interaction(on_map_click)
    btn_tablas.on_click(on_cargar_tablas)
    btn_actualizar.on_click(refrescar_capas_anio)

    ambito = f'Región {REGION_ID}' if REGION_ID is not None else 'Nacional'
    modo_txt = {
        'todas': 'bosque + plantaciones + otras',
        'bosque_plantacion': 'bosque + plantaciones',
    }.get(modo_mapa, 'bosque')

    header = widgets.HTML(
        '<div style="font-family:system-ui;font-size:14px;margin:4px 0 8px 0;color:#1a202c">'
        f'<b>{ambito}</b> · clasificación original · anomalías: <b>{modo_txt}</b>'
        '</div>'
    )

    m_orig.add_legend(
        title='Ventana temporal',
        legend_dict=LEGEND_VENTANA,
        position='bottomleft',
    )
    if REGION_ID is None:
        m_orig.fit_bounds([[-4.5, -82.0], [13.5, -66.5]])
    else:
        m_orig.set_center(-73.0, 4.5, 7)

    panels = widgets.VBox([
        _section_title('Mapa · clasificación original', ambito),
        header,
        controles,
        status_html,
        m_orig,
        _section_title('Serie del píxel · original vs corregida'),
        chart_box,
        _section_title('Patrones de trayectoria'),
        btn_tablas,
        table_html,
    ], layout=widgets.Layout(width='100%'))

    clear_output(wait=True)
    display(panels)
    mostrar_placeholder_pixel()
    refrescar_capas_anio()
